In [16]:
from sqlalchemy import String, Integer, DateTime, Date, text, Boolean, JSON, Float
import pandas as pd
from datetime import datetime
import logging
from typing import List
from statsbombpy import sb
import numpy as np
import math

In [17]:
# [2026-08-11 20:47:31,832] INFO - BRONZE - Writing 3869486 into bronze.lineups
# [2026-08-11 20:47:31,849] WARNING - BRONZE - Values that haven't been accounted for : {'Substitution - On (Tactical)', 'Player Off', 'Player On'}

In [23]:
def extract_cards( row, card_type):
    """
    Doc String
    """
    cards_values = row['cards']
    try:
        card_dict = dict(cards_values)
        if card_type == card_dict['card_type']:
            return (card_dict['time'], card_dict['reason'], card_dict['period'])
    except:
        pass

    return (None, None, None)

def extract_positions( row):
    """
    Doc String
    """
    positions_values = row['positions']
    try:
        position_dict = dict(positions_values)
        return (
            position_dict['position_id'], 
            position_dict['position'], 
            position_dict['from'],
            position_dict['to'], 
            position_dict['from_period'], 
            position_dict['to_period'],
            position_dict['start_reason'], 
            position_dict['end_reason']
        )
    except:
        pass

    return (
            None,
            None,
            None,
            None,
            None,
            None,
            None,
            None
        )

def start_and_end_game( row):
    """
    Doc String
    """
    start_reason = str(row['start_reason']) if row['start_reason'] is not None else ''
    end_reason = str(row['end_reason']) if row['end_reason'] is not None else ''

    starting_xi = 1 if start_reason == 'Starting XI' else None
    sub_on = row['from'] if "Substitution - On" in start_reason else None
    sub_off = row['to'] if "Substitution - Off" in end_reason else None
    removed_due_to_red_card = row['to'] if end_reason == 'Foul Committed (Red Card)' else None
    ended_game = 1 if end_reason == 'Final Whistle' else None

    return (starting_xi, sub_off, sub_on, removed_due_to_red_card, ended_game)


all_lineups = sb.lineups(match_id = 3857278)

teams = list(all_lineups.keys())

teams_lineups = []

for team in teams:
    teams_lineups.append(all_lineups[team])

lineups = pd.concat(teams_lineups)

lineups.insert(0, 'match_id', 3869486)

lineups['player_nickname'] = lineups['player_nickname'].fillna(lineups['player_name'])

cards = lineups.explode('cards')
cards = cards[['player_id', 'player_nickname', 'cards']]
cards['cards'] = cards['cards'].fillna(None)

cards[['first_yellow_time', 'first_yellow_reason', 'first_yellow_period']] = cards.apply(lambda x: extract_cards(x, 'Yellow Card'), axis=1, result_type='expand')
cards[['second_yellow_time', 'second_yellow_reason', 'second_yellow_period']] = cards.apply(lambda x: extract_cards(x, 'Second Yellow'), axis=1, result_type='expand')
cards[['red_card_time', 'red_card_reason', 'red_card_period']] = cards.apply(lambda x: extract_cards(x, 'Red Card'), axis=1, result_type='expand')

cards = cards.drop('cards', axis = 1)

final_cards = cards.groupby('player_id', as_index = False).agg({
    'first_yellow_time': 'max', 
    'first_yellow_reason': 'max',	
    'first_yellow_period': 'max', 
    'second_yellow_time': 'max', 
    'second_yellow_reason': 'max', 
    'second_yellow_period': 'max',
    'red_card_time': 'max', 
    'red_card_reason': 'max', 
    'red_card_period': 'max'
    }
)


new_lineups = lineups.drop('cards', axis = 1).merge(final_cards, on = 'player_id', how = 'left')


positions = new_lineups.explode('positions')
positions = positions[['player_id', 'player_nickname', 'positions']]
positions['positions'] = positions['positions'].fillna(None)


positions[['position_id', 'position', 'from', 'to', 'from_period', 'to_period', 'start_reason', 'end_reason']] = positions.apply(lambda x: extract_positions(x), axis=1, result_type='expand')

positions = positions.drop('positions', axis = 1)

starting_11 = positions["start_reason"] == "Starting XI"
sub_on = positions["start_reason"].str.contains("Substitution - On", na=False)
sub_off = positions['end_reason'].str.contains("Substitution - Off", na=False)
red_card = positions['end_reason'] == 'Foul Committed (Red Card)'
final = positions['end_reason'] == 'Final Whistle'

combined = starting_11 | sub_on | sub_off | red_card | final

positions_final = positions[combined]

positions_final[['starting_xi','sub_off','sub_on','removed_due_to_red_card','ended_game']] = positions_final.apply(lambda x: start_and_end_game(x), axis = 1,  result_type='expand').fillna(None)

grouped_positions = positions_final.groupby(['player_id'], as_index = False).agg({
    'starting_xi': 'max',
    'sub_off': 'max',
    'sub_on': 'max',
    'removed_due_to_red_card': 'max',
    'ended_game': 'max'
})

grouped_positions.loc[grouped_positions['removed_due_to_red_card'].notna(), 'ended_game'] = 0

grouped_positions['starting_xi'] = grouped_positions['starting_xi'].fillna(0)
grouped_positions['ended_game'] = grouped_positions['ended_game'].fillna(0)

final_lineups = new_lineups.drop('positions', axis = 1).merge(grouped_positions, on = 'player_id', how = 'left')

c:\Users\ellio\Documents\baller-automation-and-dashboarding-project\data_extraction_workflow\.venv\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [24]:
positions[~combined]

,player_id,player_nickname,position_id,position,from,to,from_period,to_period,start_reason,end_reason
1,3658,DeAndre Yedlin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,12746,Sean Johnson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,12757,Aaron Long,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,13442,Cristian Roldán León,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,21201,Ethan Horvath,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,22168,Josh Sargent,24.0,Left Center Forward,51:54,75:30,2.0,2.0,Tactical Shift,Player Off
19,22433,Jesus Ferreira,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,23789,Jordan Morris,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,24243,Brenden Aaronson,16.0,Left Midfield,51:54,83:14,2.0,2.0,Tactical Shift,Tactical Shift
22,33274,Luca de la Torre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
positions[positions['player_id'] == 125818]

,player_id,player_nickname,position_id,position,from,to,from_period,to_period,start_reason,end_reason
47,125818,Ali Karimi,15.0,Left Center Midfield,45:00,47:01,2.0,1.0,Tactical Shift,Substitution - On (Injury)
47,125818,Ali Karimi,6.0,Left Back,47:01,47:33,1.0,1.0,Substitution - On (Injury),Tactical Shift
47,125818,Ali Karimi,15.0,Left Center Midfield,47:33,50:30,1.0,2.0,Tactical Shift,Tactical Shift
47,125818,Ali Karimi,13.0,Right Center Midfield,50:30,62:37,2.0,2.0,Tactical Shift,Tactical Shift
47,125818,Ali Karimi,15.0,Left Center Midfield,62:37,78:10,2.0,2.0,Tactical Shift,Tactical Shift
47,125818,Ali Karimi,11.0,Left Defensive Midfield,78:10,81:30,2.0,2.0,Tactical Shift,Tactical Shift
47,125818,Ali Karimi,9.0,Right Defensive Midfield,81:30,NaN,2.0,NaN,Tactical Shift,Final Whistle


In [21]:
final_lineups

,match_id,player_id,player_name,player_nickname,jersey_number,country,first_yellow_time,first_yellow_reason,first_yellow_period,second_yellow_time,second_yellow_reason,second_yellow_period,red_card_time,red_card_reason,red_card_period,starting_xi,sub_off,sub_on,removed_due_to_red_card,ended_game
0,3869486,2941,Ismaïla Sarr,Ismaïla Sarr,18,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,73:57,NaN,NaN,0.0
1,3869486,3404,Youssouf Sabaly,Youssouf Sabaly,21,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
2,3869486,3436,Idrissa Gana Gueye,Idrissa Gueye,5,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
3,3869486,3675,Fodé Ballo Touré,Fodé Ballo Touré,12,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3869486,3684,Cheikhou Kouyaté,Cheikhou Kouyaté,8,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3869486,4506,Nampalys Mendy,Nampalys Mendy,6,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,76:58,NaN,NaN,0.0
6,3869486,5675,Kalidou Koulibaly,Kalidou Koulibaly,3,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
7,3869486,7379,Edouard Mendy,Edouard Mendy,16,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
8,3869486,8553,Abdou Diallo,Abdou Diallo,22,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
9,3869486,8870,Alfred Benjamin Gomis,Alfred Gomis,23,Senegal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
